# Snowball v6 PIT (P0-6 Split-Immune) — US + TW 2,000-Stock Kaggle Training & Signal Export

Upgraded end-to-end Kaggle training notebook incorporating **all P0-1 through P0-6 audit fixes**:

- **P0-6 Split-Immune Prices & Stale-Filing Expiration**:
  - Downloads `auto_adjust=False` to preserve unadjusted nominal `raw_close_usd` alongside total-return `close_usd`.
  - Fetches **US SEC EDGAR XBRL** (`filed + 1d`, intra-TTM split-normalized EPS, `rule40_pct`, `gross_margin_chg`, `roe`, `ep_yield`) and **TW TWSE/TPEx** monthly snapshots (`ep_yield`, `roe`).
  - Strictly expires stale filings (`max_staleness_days=180`, `max_period_age_days=270`).
- **Economic Monotonicity & Sign Constraints**:
  - Constrained Ridge (`L-BFGS-B`) + Monotone LightGBM (`max_depth=3`, `num_leaves=7`, `reg_lambda=50`), preventing bull-market high-volatility chasing (`f_vol_60d <= 0`).
- **3 Dedicated Purged Walk-Forward Runs (Strict Purge Gap `>= horizon_weeks`)**:
  1. **Combined US + TW (2,000 stocks, 4w)** — full cross-market benchmark
  2. **US-Specialized (1,000 stocks, 12w)** — captures quarterly PIT Value (`E/P`) & fundamental alpha
  3. **TW-Specialized (1,000 stocks, 4w)** — captures Taiwan Residual Momentum, Industry Momentum & ROE
- **Zero Train-Serve Skew Export (P0-1 Ready)**:
  - Exports `v6_learned_weights.json`, `v6_learned_weights_us.json`, `v6_learned_weights_tw.json`, PLUS ready-to-trade `signals_us_latest.csv` & `signals_tw_latest.csv` directly accepted by `broker_exec.py --mode paper --signals-csv`!

> **Kaggle Settings**: Enable **Internet: ON** (needed to clone Snowball, download Yahoo `raw_close_usd`, SEC EDGAR XBRL & TWSE/TPEx). Click **Save Version → Save & Run All** so `/kaggle/working/snowball_artifacts/` and `snowball_us_tw_2000_pit.parquet` are saved to Output.

In [ ]:
import os, sys, subprocess, platform
from pathlib import Path

REPO_URL = "https://github.com/948787878787/snowball.git"
IS_KAGGLE = Path("/kaggle/working").exists()
WORK_DIR = Path("/kaggle/working") if IS_KAGGLE else Path.cwd()

# Always pull latest main (includes P0-1..P0-6 fixes) when on Kaggle
repo_dir = WORK_DIR / "snowball"
if IS_KAGGLE:
    if repo_dir.exists():
        subprocess.check_call(["git", "-C", str(repo_dir), "pull", "--ff-only"])
    else:
        subprocess.check_call(["git", "clone", "--depth", "1", REPO_URL, str(repo_dir)])
    SCRIPTS_DIR = repo_dir / "skills" / "snowball" / "scripts"
else:
    local_candidates = [
        Path.cwd().parent / "skills" / "snowball" / "scripts",
        Path.cwd() / "skills" / "snowball" / "scripts",
        repo_dir / "skills" / "snowball" / "scripts",
    ]
    SCRIPTS_DIR = next((p for p in local_candidates if p.exists()), None)
    if SCRIPTS_DIR is None:
        subprocess.check_call(["git", "clone", "--depth", "1", REPO_URL, str(repo_dir)])
        SCRIPTS_DIR = repo_dir / "skills" / "snowball" / "scripts"

if str(SCRIPTS_DIR) not in sys.path:
    sys.path.insert(0, str(SCRIPTS_DIR))

for pkg in ("yfinance", "lightgbm", "pyarrow"):
    try:
        __import__(pkg)
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

print("Environment :", "Kaggle" if IS_KAGGLE else platform.platform())
print("Scripts Dir :", SCRIPTS_DIR)
print("CPU Cores   :", os.cpu_count())

In [ ]:
# === Training Configuration (P0-6 Split-Immune US 1,000 + TW 1,000) ===
MARKETS = ("US", "TW")
HISTORY_PERIOD = "8y"          # 8 years covering 2020 COVID, 2022 bear, 2023-2026 bull
FETCH_PIT_FUNDAMENTALS = True   # Fetch free SEC EDGAR XBRL (US) + TWSE/TPEx (TW)
MAX_STALENESS_DAYS = 180        # P0-6: expire stale filings >180d
MIN_TRAIN_WEEKS = 104           # 2-year minimum expanding warmup
REFIT_STEP_WEEKS = 8            # Bi-monthly expanding refits
RIDGE_ALPHA = 500.0             # Strong L2 shrinkage
TOP_K_PORTFOLIO = 40            # Top-40 combined (Top-20 per single market)
USE_GPU = False                 # Optional: True if Kaggle GPU T4/P100 is attached
OFFLINE_DEMO = False            # True for <15s synthetic offline sanity check

ARTIFACT_DIR = WORK_DIR / "snowball_artifacts"
PIT_CACHE_DIR = WORK_DIR / "pit_cache"
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
PIT_CACHE_DIR.mkdir(parents=True, exist_ok=True)
print("Artifacts will be saved to:", ARTIFACT_DIR)

## 1. Build Split-Immune 2,000-Stock Panel (`raw_close_usd` + `close_usd`) & Merge PIT Fundamentals
If a mounted Kaggle dataset already contains `raw_close_usd` and `pit_ep_z`, it loads immediately. Otherwise, this cell rebuilds the split-immune panel (`auto_adjust=False` for exact `raw_close_usd`) and merges SEC EDGAR XBRL + TWSE/TPEx Point-in-Time fundamentals with 180-day staleness expiration.

In [ ]:
import pandas as pd
from build_training_panel import build_combined_panel, load_dataframe, save_dataframe
from fetch_pit_fundamentals import fetch_us_sec_pit, fetch_tw_exchange_pit, merge_pit_into_panel

pit_panel_path = WORK_DIR / "snowball_us_tw_2000_pit.parquet"
cached_candidates = []
if Path("/kaggle/input").exists():
    cached_candidates.extend(sorted(Path("/kaggle/input").rglob("snowball_us_tw_2000_pit.*")))
cached_candidates.append(pit_panel_path)

panel = None
if not OFFLINE_DEMO:
    for cand in cached_candidates:
        if cand.exists():
            tmp = load_dataframe(cand)
            if "raw_close_usd" in tmp.columns and "pit_ep_z" in tmp.columns:
                print(f"Loaded P0-6 split-immune PIT panel: {cand}")
                panel = tmp
                break
            else:
                print(f"[INFO] Skipping legacy panel {cand} (missing raw_close_usd -> rebuilding split-immune P0-6 panel).")

if panel is None:
    print(f"Building split-immune {MARKETS} panel ({'OFFLINE DEMO' if OFFLINE_DEMO else HISTORY_PERIOD + ' LIVE'})...")
    base_panel = build_combined_panel(
        markets=MARKETS,
        period=HISTORY_PERIOD,
        horizons=(4, 12),
        offline_demo=OFFLINE_DEMO,
        demo_stocks_per_market=120 if OFFLINE_DEMO else None,
        batch_size=150,
    )
    assert "raw_close_usd" in base_panel.columns, "P0-6 requires raw_close_usd!"

    if FETCH_PIT_FUNDAMENTALS and not OFFLINE_DEMO:
        us_tickers = sorted(base_panel.loc[base_panel["market"] == "US", "ticker"].unique().tolist())
        tw_tickers = sorted(base_panel.loc[base_panel["market"] == "TW", "ticker"].unique().tolist())
        start_str = (base_panel["date"].min() - pd.Timedelta(days=60)).strftime("%Y-%m-%d")
        end_str = base_panel["date"].max().strftime("%Y-%m-%d")

        pit_parts = []
        if us_tickers:
            df_us = fetch_us_sec_pit(us_tickers, cache_dir=PIT_CACHE_DIR, min_year=max(2017, base_panel["date"].min().year - 1))
            if not df_us.empty:
                pit_parts.append(df_us)
        if tw_tickers:
            df_tw = fetch_tw_exchange_pit(tw_tickers, cache_dir=PIT_CACHE_DIR, start_date=start_str, end_date=end_str)
            if not df_tw.empty:
                pit_parts.append(df_tw)

        if pit_parts:
            pit_df = pd.concat(pit_parts, ignore_index=True)
            save_dataframe(pit_df, WORK_DIR / "pit_fundamentals_us_tw.parquet")
            panel = merge_pit_into_panel(base_panel, pit_df, max_staleness_days=MAX_STALENESS_DAYS)
        else:
            panel = base_panel
    else:
        panel = base_panel

    save_dataframe(panel, pit_panel_path)
    print(f"Saved P0-6 split-immune PIT panel: {pit_panel_path}")

print(f"\nPanel Shape : {panel.shape}")
print(f"Date Range  : {panel['date'].min().date()} -> {panel['date'].max().date()} ({panel['date'].nunique()} weeks)")
print("Tickers/Mkt :", panel.groupby("market")["ticker"].nunique().to_dict())
for c in [col for col in panel.columns if col.startswith("pit_")]:
    u = panel.loc[panel["market"] == "US", c].notna().mean() * 100
    t = panel.loc[panel["market"] == "TW", c].notna().mean() * 100
    print(f"  {c:<24}: US={u:5.1f}% | TW={t:5.1f}%")

## 2. Run 3 Strict Purged Walk-Forward Models (Combined 2,000 + US-12W Specialized + TW-4W Specialized)
Separates US (quarterly 12w horizon, heavy PIT Value) and TW (4w horizon, heavy Residual & Industry Momentum) alongside the combined 2,000-stock benchmark.

In [ ]:
import shutil
from train_walkforward import (
    run_purged_walkforward,
    export_training_artifacts,
    format_metrics_table,
)

eff_min_train = min(MIN_TRAIN_WEEKS, max(36, panel["date"].nunique() // 3))

# 1. Combined US + TW (2,000 stocks, 4w)
res_all = run_purged_walkforward(
    panel=panel, horizon=4, min_train_weeks=eff_min_train,
    refit_step_weeks=REFIT_STEP_WEEKS, ridge_alpha=RIDGE_ALPHA,
    enable_lgbm=True, use_gpu=USE_GPU, top_k=TOP_K_PORTFOLIO,
)
export_training_artifacts(res_all, ARTIFACT_DIR)
print("=== [1] COMBINED US + TW (4W) OOS ===")
display(format_metrics_table(res_all["metrics"]).round(4))
print("Learned Combined v6 Weights:", res_all["learned_v6_weights"])

# 2. US-Specialized (1,000 US stocks, 12w quarterly horizon)
panel_us = panel.loc[panel["market"] == "US"].copy()
res_us = run_purged_walkforward(
    panel=panel_us, horizon=12, min_train_weeks=eff_min_train,
    refit_step_weeks=REFIT_STEP_WEEKS, ridge_alpha=RIDGE_ALPHA,
    enable_lgbm=True, use_gpu=USE_GPU, top_k=20,
)
export_training_artifacts(res_us, ARTIFACT_DIR / "us_12w")
shutil.copy(ARTIFACT_DIR / "us_12w" / "v6_learned_weights.json", ARTIFACT_DIR / "v6_learned_weights_us.json")
print("\n=== [2] US-SPECIALIZED (12W) OOS ===")
display(format_metrics_table(res_us["metrics"]).round(4))
print("Learned US v6 Weights:", res_us["learned_v6_weights"])

# 3. TW-Specialized (1,000 TW stocks, 4w horizon)
panel_tw = panel.loc[panel["market"] == "TW"].copy()
res_tw = run_purged_walkforward(
    panel=panel_tw, horizon=4, min_train_weeks=eff_min_train,
    refit_step_weeks=REFIT_STEP_WEEKS, ridge_alpha=RIDGE_ALPHA,
    enable_lgbm=True, use_gpu=USE_GPU, top_k=20,
)
export_training_artifacts(res_tw, ARTIFACT_DIR / "tw_4w")
shutil.copy(ARTIFACT_DIR / "tw_4w" / "v6_learned_weights.json", ARTIFACT_DIR / "v6_learned_weights_tw.json")
print("\n=== [3] TW-SPECIALIZED (4W) OOS ===")
display(format_metrics_table(res_tw["metrics"]).round(4))
print("Learned TW v6 Weights:", res_tw["learned_v6_weights"])

## 3. Export Production Signals CSV (`signals_us_latest.csv` & `signals_tw_latest.csv`) — P0-1 Zero Train-Serve Skew
Scores the latest week using exact learned weights + real PIT fundamentals and outputs nominal prices (`raw_close_usd` converted to TWD for TW) so `broker_exec.py --mode paper --signals-csv` executes with zero feature skew.

In [ ]:
import numpy as np
import yfinance as yf

latest_dt = panel["date"].max()
latest = panel.loc[panel["date"] == latest_dt].copy()

try:
    fx_hist = yf.Ticker("TWD=X").history(period="5d")
    twd_per_usd = float(fx_hist["Close"].dropna().iloc[-1]) if not fx_hist.empty else 32.0
except Exception:
    twd_per_usd = 32.0

def _export_market_signals(sub_df: pd.DataFrame, weights: dict, market: str, out_csv: Path) -> pd.DataFrame:
    df = sub_df.copy()
    score = (
        weights["mom_qual"] * df["f_mom_qual"].fillna(0.0)
        + weights["ind_mom"] * df["f_ind_mom"].fillna(0.0)
        + weights["norm_val"] * df.get("pit_ep_z", pd.Series(0.0, index=df.index)).fillna(0.0)
        + weights["quality"] * (
            0.50 * df.get("pit_roe_z", pd.Series(0.0, index=df.index)).fillna(0.0)
            + 0.25 * df.get("pit_gross_margin_chg_z", pd.Series(0.0, index=df.index)).fillna(0.0)
            + 0.25 * df.get("pit_fcf_margin_z", pd.Series(0.0, index=df.index)).fillna(0.0)
        )
        + weights["fund"] * (
            0.60 * df.get("pit_rule40_z", pd.Series(0.0, index=df.index)).fillna(0.0)
            + 0.40 * df.get("pit_rev_yoy_z", pd.Series(0.0, index=df.index)).fillna(0.0)
        )
        + weights["low_risk"] * df["f_low_risk"].fillna(0.0)
    )
    sd = float(score.std())
    df["v6_score"] = ((score - float(score.mean())) / sd).clip(-3.0, 3.0) if sd > 1e-12 else 0.0
    px_col = "raw_close_usd" if "raw_close_usd" in df.columns else "close_usd"
    df["close"] = df[px_col] * (twd_per_usd if market == "TW" else 1.0)
    out = df[["ticker", "industry", "close", "v6_score"]].dropna().sort_values("v6_score", ascending=False)
    out.to_csv(out_csv, index=False)
    return out

sig_us = _export_market_signals(latest[latest["market"] == "US"], res_us["learned_v6_weights"], "US", ARTIFACT_DIR / "signals_us_latest.csv")
sig_tw = _export_market_signals(latest[latest["market"] == "TW"], res_tw["learned_v6_weights"], "TW", ARTIFACT_DIR / "signals_tw_latest.csv")

print(f"Exported P0-1 ready signals (date={latest_dt.date()}, TWD/USD={twd_per_usd:.2f}):")
print("Top 10 US:")
display(sig_us.head(10).round(3))
print("Top 10 TW:")
display(sig_tw.head(10).round(3))

## 4. OOS Diagnostics & Visualizations

In [ ]:
import matplotlib.pyplot as plt
from train_walkforward import _compute_daily_ic_series

fig, axes = plt.subplots(1, 3, figsize=(18, 4.8))

for ax, res_obj, title, h in [
    (axes[0], res_all, "Combined US+TW (4W) Cum. OOS IC", 4),
    (axes[1], res_us,  "US-Specialized (12W) Cum. OOS IC", 12),
    (axes[2], res_tw,  "TW-Specialized (4W) Cum. OOS IC", 4),
]:
    oos_df = res_obj["oos_predictions"]
    tcol = f"target_rank_{h}w"
    for m in res_obj["metrics"]:
        ic_s = _compute_daily_ic_series(oos_df, f"pred_{m}", tcol)
        ax.plot(ic_s.index, ic_s.cumsum(), label=f"{m} (IC={ic_s.mean():.3f})", linewidth=2)
    ax.axhline(0, color="gray", linestyle="--", linewidth=0.8)
    ax.set_title(title)
    ax.legend(fontsize=8)
    ax.grid(alpha=0.25)

plt.tight_layout()
plt.savefig(ARTIFACT_DIR / "oos_diagnostics.png", dpi=150)
plt.show()

## 5. Deploy Back to Local Paper Trading (`broker_exec.py` + `global_ranker.py`)
Download `/kaggle/working/snowball_artifacts/` from Kaggle's **Output** tab and run:

```bash
# Live ranking with market-specialized learned weights:
python skills/snowball/scripts/global_ranker.py --market US --learned-weights ./v6_learned_weights_us.json --top 20
python skills/snowball/scripts/global_ranker.py --market TW --learned-weights ./v6_learned_weights_tw.json --top 20

# Safe Paper Trading (P0-1 zero Train-Serve Skew via --signals-csv):
python skills/snowball/scripts/broker_exec.py --broker alpaca --mode paper --cash 1500 --signals-csv ./signals_us_latest.csv
python skills/snowball/scripts/broker_exec.py --broker shioaji --mode paper --cash 50000 --signals-csv ./signals_tw_latest.csv
```